# ViGoEmotions — Kaggle: 28 nhãn, gồm Neutral

So sánh TF-IDF + OvR LR, BiGRU và PhoBERT với baseline 27 nhãn.
Attach hai Dataset Private: ZIP code 28 nhãn và ba CSV train/val/test.
Trong Settings/Session options, bật GPU + Internet; không chọn TPU.
Chạy từng cell từ trên xuống; dừng khi có lỗi. Hướng dẫn: docs/neutral_28_labels.md.
Artifact ở /kaggle/working/ViGoEmotions28/reports/neutral_28_labels/.
Model hoàn tất được giữ; không hỗ trợ resume giữa epoch.


## 1. Nạp project từ Kaggle Input

Attach Dataset chứa **vigoemotions_neutral28_colab.zip** mới, không dùng ZIP 04/05.
Hỗ trợ cả ZIP và Dataset đã được Kaggle giải nén. Code tự tìm đường dẫn.
Cell dưới chỉ chạy một lần trong workspace mới; không giải nén đè kết quả.
Sau restart kernel, bỏ qua cell giải nén nếu project còn và chạy cell nối workspace.


In [2]:
from pathlib import Path
import os, shutil, zipfile

INPUT = Path('/kaggle/input')
ROOT = Path('/kaggle/working/ViGoEmotions28')
if ROOT.exists():
    raise FileExistsError(f'{ROOT} đã tồn tại. Không giải nén đè lên kết quả; chạy tiếp cell cần thiết hoặc dùng session mới.')

# Kaggle có thể giữ ZIP hoặc tự giải nén ZIP khi tạo Dataset.
archives = list(INPUT.rglob('vigoemotions_neutral28_colab.zip'))
code_roots = [p.parent.parent for p in INPUT.rglob('scripts/neutral_28_labels.py')]
if len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as archive:
        for item in archive.infolist():
            if not (ROOT / item.filename).resolve().is_relative_to(ROOT.resolve()):
                raise ValueError('Unsafe ZIP path')
        ROOT.mkdir(parents=True)
        archive.extractall(ROOT)
elif not archives and len(code_roots) == 1:
    shutil.copytree(code_roots[0], ROOT)
else:
    raise RuntimeError('Hãy attach đúng một Dataset code chứa vigoemotions_neutral28_colab.zip hoặc project đã giải nén.')
os.chdir(ROOT)
print('Project:', ROOT)


Project: /kaggle/working/ViGoEmotions28


In [4]:
# Cũng dùng cell này để nối lại workspace sau khi restart kernel.
from pathlib import Path
import os, sys, subprocess
ROOT = Path('/kaggle/working/ViGoEmotions28')
assert (ROOT/'scripts/neutral_28_labels.py').is_file(), 'Thiếu module 28 nhãn; kiểm tra ZIP/version repo.'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'scripts'))


## 2. Dependency tối thiểu và GPU
Không cài bộ lock local của macOS. Không cần reportlab/pypdf vì báo cáo thí nghiệm là Markdown.
Nếu pip yêu cầu restart kernel, restart rồi chạy lại cell nối workspace, GPU, helper và các bước sau;
không cần upload/giải nén lại. Các lệnh cài đặt có kiểm tra exit code.


In [5]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'numpy>=1.26,<3', 'pandas>=2.2,<4', 'scipy>=1.13,<2', 'scikit-learn>=1.6,<2',
    'matplotlib>=3.9,<4', 'torch>=2.6,<3', 'transformers==4.57.3',
    'py-vncorenlp==0.1.4', 'joblib', 'threadpoolctl', 'tqdm'], check=True)
import shutil
if shutil.which('java') is None:
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'default-jre-headless'], check=True)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [6]:
import torch, subprocess
print('Python:', sys.version)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('TF-IDF vẫn chạy được; bật GPU trước BiGRU/PhoBERT.')
subprocess.run(['java', '-version'], check=True)


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
CUDA available: True
GPU: Tesla T4


openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


CompletedProcess(args=['java', '-version'], returncode=0)

## 3. Dữ liệu và quy tắc annotation
Attach Dataset Private chứa `train.csv`, `val.csv`, `test.csv` có quyền truy cập. Không đưa dữ liệu nguồn vào repo công khai.
Giữ split gốc; áp dụng cùng quy tắc loại train trùng heldout của pipeline 27 nhãn (không tạo split mới).
Test text chỉ được dùng để audit overlap như code gốc; không dùng test target để fit/chọn epoch/ngưỡng.

**Target 28 chiều:** `[27]` → chỉ Neutral = 1; `[2,27]` → Joy = 1 và Neutral = 1.
Giữ đầy đủ nhãn gốc; ID lặp gộp khi multi-hot. Pipeline 27 nhãn cũ không thay đổi.
Guideline do người dùng cung cấp nói Neutral không đi cùng cảm xúc, nhưng dữ liệu có **306 mixed**
và **725 Neutral-only** (tổng trước loại overlap). Chúng ta giữ annotation vì chưa biết nhãn nào đúng.
Không tự sửa hay loại các mixed sample. Nhánh ablation bổ sung không phải điều kiện hoàn thành study này.


In [8]:
from pathlib import Path
import csv, shutil

sources = {}
for name in ('train.csv', 'val.csv', 'test.csv'):
    candidates = []
    for path in Path('/kaggle/input').rglob(name):
        with path.open(encoding='utf-8-sig', newline='') as handle:
            header = next(csv.reader(handle), None)
        # ZIP code có splits/*.csv chỉ chứa ID, không phải dữ liệu nguồn.
        if path.parent.name == 'splits' and header == ['id']:
            continue
        if header != ['id', 'text', 'labels']:
            raise ValueError(f'{path}: schema phải là id,text,labels')
        candidates.append(path)
    if len(candidates) != 1:
        raise RuntimeError(f'Cần đúng một {name} dữ liệu (id,text,labels) trong Input; tìm thấy {len(candidates)}. Kiểm tra Dataset đã attach.')
    sources[name] = candidates[0]
(ROOT / 'data').mkdir(exist_ok=True)
for name, source in sources.items():
    shutil.copy2(source, ROOT / 'data' / name)
    print('Đã nạp:', name)

DATA = ROOT / 'data'


Đã nạp: train.csv
Đã nạp: val.csv
Đã nạp: test.csv


## 4. Helper và tạo study riêng
Sao chép settings từ `configs/midterm.json` đi kèm ZIP; kiểm tra epoch trong config in ra sau init.
Config hiện tại có thể khác lượt baseline 27 nhãn đã lưu; không tự lấy epoch từ notebook khác.
Nếu baseline bạn muốn đối chiếu dùng epoch khác, hãy cung cấp đúng config/run/report của baseline đó
và tạo **study mới**; không sửa config gốc. Điều kiện khác môi trường, epoch và checksum được ghi trong bảng.
Từng model lưu toàn bộ config thực dùng, environment, thời gian và checkpoint selection.


In [9]:
import neutral_28_labels as n28
from IPython.display import display
import pandas as pd
import json

OUT = ROOT/'reports/neutral_28_labels'
BASE_CONFIG = ROOT/'configs/midterm.json'
BASELINE_REPORTS = ROOT/'reports'  # Có thể trỏ đến reports 27 nhãn khác, chỉ đọc.

# Có thể đổi OUT thành ROOT/'reports/neutral_28_labels/study_moi' trước khi init.
# Các lựa chọn được đóng băng khi init, không sửa giữa study.
def run28(action, *arguments):
    subprocess.run([sys.executable, '-u', str(ROOT/'scripts/neutral_28_labels.py'), action,
                    '--output', str(OUT), '--data-dir', str(DATA), *map(str, arguments)],
                   cwd=ROOT, check=True)

run28('init', '--base-config', BASE_CONFIG, '--baseline-reports', BASELINE_REPORTS)
display(pd.read_csv(OUT/'annotation_counts.csv'))
print('Config:', json.loads((OUT/'config.json').read_text()))
assert n28.LABELS28[27] == 'neutral' and len(n28.LABELS28) == 28
assert n28.multihot28([27]).tolist() == [0]*27 + [1]
assert n28.multihot28([2,27])[2] == n28.multihot28([2,27])[27] == 1


Seed: 42 | labels: 28 | threshold: 0.5
split  raw_samples  neutral_only  mixed_neutral  neutral_positive
train        16531           570            251               821
  val         2066            75             19                94
 test         2067            80             36               116
Excluded train overlaps: 1
No 28-label models trained yet.


,split,raw_samples,neutral_only,mixed_neutral,neutral_positive
0,train,16531,570,251,821
1,val,2066,75,19,94
2,test,2067,80,36,116


Config: {'seed': 42, 'label_count': 28, 'neutral_id': 27, 'threshold': 0.5, 'split_policy': 'Preserve supplied val/test; remove normalized train overlaps; audit label prevalence', 'frequency_tiers': {'method': 'descending training frequency rank; tie by label ID', 'head_ranks': [1, 9], 'mid_ranks': [10, 18], 'tail_ranks': [19, 27]}, 'tfidf': {'max_features': 40000, 'ngram_range': [1, 2], 'min_df': 2, 'sublinear_tf': True, 'lowercase': True, 'token_pattern': '(?u)\\b\\w+\\b'}, 'logistic_regression': {'C': 1.0, 'solver': 'liblinear', 'max_iter': 1000, 'random_state': 42}, 'gru': {'max_vocab': 30000, 'embedding_dim': 128, 'hidden_dim': 128, 'dropout': 0.3, 'max_length': 128, 'epochs': 10, 'batch_size': 64, 'learning_rate': 0.001}, 'phobert': {'checkpoint': 'vinai/phobert-base-v2', 'max_length': 256, 'epochs': 12, 'batch_size': 16, 'learning_rate': 2e-05, 'weight_decay': 0.01, 'revision': '86cd7fd4c148980922ac11a2cf5e257f2ba639e1'}, 'checkpoint_selection': 'Highest validation 28-label Macr

## 5. Smoke tests offline
Kiểm tra target/metrics, fit TF-IDF và BiGRU nhỏ, backward/nạp lại checkpoint và một RoBERTa tiny
cùng kiểu head với PhoBERT. Mô hình tiny chỉ kiểm tra kỹ thuật, không phải pretrained PhoBERT và không
được ghi vào kết quả thí nghiệm. Test này không phụ thuộc config 4 epoch cố định của bộ test cũ.


In [10]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
                '-p', 'test_neutral_28_labels.py', '-v'], cwd=ROOT, check=True)


test_configuration_changes_cannot_silently_reuse_existing_study (test_neutral_28_labels.Neutral28Tests.test_configuration_changes_cannot_silently_reuse_existing_study) ... ok
test_gru_real_backward_checkpoint_and_reload (test_neutral_28_labels.Neutral28Tests.test_gru_real_backward_checkpoint_and_reload) ... /usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.13/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enab

Device: cuda | seed: 42 | outputs: 28


gru epoch 1/1: 100%|██████████| 2/2 [00:02<00:00,  1.30s/it]
ok
test_loader_preserves_annotations_and_removes_only_train_overlap (test_neutral_28_labels.Neutral28Tests.test_loader_preserves_annotations_and_removes_only_train_overlap) ... ok
test_metrics_include_neutral_and_emotion_projection (test_neutral_28_labels.Neutral28Tests.test_metrics_include_neutral_and_emotion_projection) ... ok
test_neural_train_interface_has_no_test_argument (test_neutral_28_labels.Neutral28Tests.test_neural_train_interface_has_no_test_argument) ... ok
test_output_cannot_target_original_reports (test_neutral_28_labels.Neutral28Tests.test_output_cannot_target_original_reports) ... ok
test_phobert_tiny_offline_full_finetune_and_reload (test_neutral_28_labels.Neutral28Tests.test_phobert_tiny_offline_full_finetune_and_reload) ... 

{'epoch': 1, 'loss': 0.697115570306778, 'macro_f1': 0.03214285714285715, 'micro_f1': 0.07272727272727272, 'hamming_loss': 0.45535714285714285, 'lrap': 0.07063943001443002}


I0000 00:00:1793795110.439834     194 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Device: cuda | seed: 42 | outputs: 28
{'epoch': 1, 'loss': 0.6937680244445801, 'macro_f1': 0.03809523809523809, 'micro_f1': 0.08695652173913043, 'hamming_loss': 0.5625, 'lrap': 0.07994505494505494}


phobert epoch 1/1: 100%|██████████| 2/2 [00:00<00:00, 12.58it/s]
ok
test_prediction_categories_and_top5_come_from_probabilities (test_neutral_28_labels.Neutral28Tests.test_prediction_categories_and_top5_come_from_probabilities) ... ok
test_reject_invalid_annotations (test_neutral_28_labels.Neutral28Tests.test_reject_invalid_annotations) ... ok
test_targets_preserve_original_and_27_pipeline_unchanged (test_neutral_28_labels.Neutral28Tests.test_targets_preserve_original_and_27_pipeline_unchanged) ... ok
test_test_evaluation_requires_all_models_and_prevents_future_training (test_neutral_28_labels.Neutral28Tests.test_test_evaluation_requires_all_models_and_prevents_future_training) ... ok
test_tfidf_fit_and_saved_reload_match (test_neutral_28_labels.Neutral28Tests.test_tfidf_fit_and_saved_reload_match) ... ok
test_validation_demo_finds_all_four_categories_without_exporting_text (test_neutral_28_labels.Neutral28Tests.test_validation_demo_finds_all_four_categories_without_exporting_text) ...

tfidf_lr neutral_only : {'model': 'tfidf_lr', 'category': 'neutral_only', 'available_count': 1, 'sample_id': 'x0'}
tfidf_lr single_emotion : {'model': 'tfidf_lr', 'category': 'single_emotion', 'available_count': 1, 'sample_id': 'x1'}
tfidf_lr multi_emotion : {'model': 'tfidf_lr', 'category': 'multi_emotion', 'available_count': 1, 'sample_id': 'x2'}
tfidf_lr neutral_and_emotion : {'model': 'tfidf_lr', 'category': 'neutral_and_emotion', 'available_count': 1, 'sample_id': 'x3'}


CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_neutral_28_labels.py', '-v'], returncode=0)

## 6. Model A — TF-IDF + One-vs-Rest Logistic Regression
Giữ nguyên TF-IDF, normalization và LogisticRegression của config gốc; fit vocabulary/IDF chỉ trên train.
28 classifier nhị phân độc lập, bao gồm Neutral. Đánh giá validation tại 0.5; lưu cả vectorizer và model
để inference không fit lại. Artifact riêng ở `models/tfidf_lr/` bên trong OUT.


In [11]:
run28('train', '--model', 'tfidf_lr')
display(pd.read_csv(OUT/'metrics_summary.csv'))


   model split  label_count  threshold    n  macro_f1  micro_f1  hamming_loss     lrap  emotion27_macro_f1  emotion27_micro_f1  emotion27_hamming_loss  emotion27_lrap
tfidf_lr   val           28        0.5 2066   0.23104   0.26616      0.060054 0.609645            0.238825            0.271238                0.060593        0.640745


,model,split,label_count,threshold,n,macro_f1,micro_f1,hamming_loss,lrap,emotion27_macro_f1,emotion27_micro_f1,emotion27_hamming_loss,emotion27_lrap
0,tfidf_lr,val,28,0.5,2066,0.23104,0.26616,0.060054,0.609645,0.238825,0.271238,0.060593,0.640745


## 7. Model B — BiGRU
Tái dùng BiGRU/forward gốc; chỉ thay head 27 → 28. Vocabulary từ train, lowercase/whitespace,
padding/truncation như cũ. BCEWithLogitsLoss, AdamW, gradient clipping 1.0, fixed seed và shuffled DataLoader
như baseline. Chọn checkpoint bằng validation **28-label Macro-F1 tại 0.5**, hòa chọn epoch sớm nhất.
Hiện tiến độ từng batch và metric mỗi epoch. Không dùng test để chọn epoch.


In [12]:
assert torch.cuda.is_available(), 'Bật GPU runtime trước khi chạy.'
run28('train', '--model', 'gru')
display(pd.read_csv(OUT/'models/gru/history.csv'))


Device: cuda | seed: 42 | outputs: 28


gru epoch 1/10: 100%|██████████| 259/259 [00:03<00:00, 74.63it/s]


{'epoch': 1, 'loss': 0.2732851952512698, 'macro_f1': 0.0382755184813978, 'micro_f1': 0.05335291238374939, 'hamming_loss': 0.06686488729083115, 'lrap': 0.3903060923999531}


gru epoch 2/10: 100%|██████████| 259/259 [00:02<00:00, 91.19it/s]


{'epoch': 2, 'loss': 0.20947331375630063, 'macro_f1': 0.15872030212859203, 'micro_f1': 0.2158428663932657, 'hamming_loss': 0.0628025169409487, 'lrap': 0.5197423671218947}


gru epoch 3/10: 100%|██████████| 259/259 [00:02<00:00, 88.89it/s]


... (12 outputs left out)

{'epoch': 9, 'loss': 0.10484110511257946, 'macro_f1': 0.3935331512675669, 'micro_f1': 0.4539282250242483, 'hamming_loss': 0.05839441294426773, 'lrap': 0.6088786695495318}


gru epoch 10/10: 100%|██████████| 259/259 [00:02<00:00, 91.57it/s]


{'epoch': 10, 'loss': 0.09515813520044106, 'macro_f1': 0.3992265474699422, 'micro_f1': 0.45909451945988883, 'hamming_loss': 0.05886115336744572, 'lrap': 0.605070900117131}
   model split  label_count  threshold    n  macro_f1  micro_f1  hamming_loss     lrap  emotion27_macro_f1  emotion27_micro_f1  emotion27_hamming_loss  emotion27_lrap
tfidf_lr   val           28        0.5 2066  0.231040  0.266160      0.060054 0.609645            0.238825            0.271238                0.060593        0.640745
     gru   val           28        0.5 2066  0.399227  0.459095      0.058861 0.605071            0.410764            0.465944                0.059177        0.636809


,epoch,loss,macro_f1,micro_f1,hamming_loss,lrap
0,1,0.273285,0.038276,0.053353,0.066865,0.390306
1,2,0.209473,0.158720,0.215843,0.062803,0.519742
2,3,0.181930,0.239827,0.309605,0.059518,0.563849
3,4,0.164233,0.305052,0.382208,0.058343,0.583586
4,5,0.149773,0.330596,0.406894,0.057703,0.592588
5,6,0.137698,0.355152,0.427787,0.057668,0.600710
6,7,0.126049,0.366429,0.429907,0.056942,0.603120
7,8,0.114922,0.379398,0.439614,0.059276,0.600544
8,9,0.104841,0.393533,0.453928,0.058394,0.608879
9,10,0.095158,0.399227,0.459095,0.058861,0.605071


## 8. Model C — PhoBERT
Giữ pretrained checkpoint/revision của baseline, VnCoreNLP wseg trước BPE và tokenizer use_fast=False.
Head 28 logits; optimizer nhận toàn bộ backbone + head (không freeze). BCEWithLogitsLoss và ngưỡng 0.5.
Tải pretrained/segmenter lần đầu cần Internet/Java. Checkpoint và tokenizer được lưu riêng để reload.
Cảnh báo head mới khởi tạo là bình thường. Mọi metric in ra là số đo thật.


In [13]:
assert torch.cuda.is_available(), 'Bật GPU runtime trước khi chạy.'
run28('train', '--model', 'phobert')
display(pd.read_csv(OUT/'models/phobert/history.csv'))


--2026-11-04 12:26:28--  https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27412703 (26M) [application/octet-stream]
Saving to: ‘VnCoreNLP-1.2.jar’

     0K .......... .......... .......... .......... ..........  0% 1.26M 21s
    50K .......... .......... .......... .......... ..........  0% 4.62M 13s
   100K .......... .......... .......... .......... ..........  0% 2.30M 12s
   150K .......... .......... .......... .......... ..........  0% 7.04M 10s
   200K .......... .......... .......... .......... ..........  0% 7.81M 9s
   250K .......... .......... .......... .......... ..........  1% 2.96M 9s
   300K .......... .......... .......... .......... ..........  1% 13.2M 8s
   350

2026-11-04 12:26:44 INFO  WordSegmenter:24 - Loading Word Segmentation model


I0000 00:00:1793795219.438761     250 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Device: cuda | seed: 42 | outputs: 28


phobert epoch 1/12: 100%|██████████| 1034/1034 [11:56<00:00,  1.44it/s]


{'epoch': 1, 'loss': 0.2601835206448491, 'macro_f1': 0.0, 'micro_f1': 0.0, 'hamming_loss': 0.06842068870142443, 'lrap': 0.42063158183141514}


... (19 outputs left out)

{'epoch': 11, 'loss': 0.050653144720822336, 'macro_f1': 0.5746532330885304, 'micro_f1': 0.5776718315454906, 'hamming_loss': 0.05512722998202185, 'lrap': 0.712200684637673}


phobert epoch 12/12: 100%|██████████| 1034/1034 [11:57<00:00,  1.44it/s]


{'epoch': 12, 'loss': 0.04395974186185411, 'macro_f1': 0.5629885944512661, 'micro_f1': 0.5696969696969697, 'hamming_loss': 0.0552309500760614, 'lrap': 0.7036056192724216}
   model split  label_count  threshold    n  macro_f1  micro_f1  hamming_loss     lrap  emotion27_macro_f1  emotion27_micro_f1  emotion27_hamming_loss  emotion27_lrap
tfidf_lr   val           28        0.5 2066  0.231040  0.266160      0.060054 0.609645            0.238825            0.271238                0.060593        0.640745
     gru   val           28        0.5 2066  0.399227  0.459095      0.058861 0.605071            0.410764            0.465944                0.059177        0.636809
 phobert   val           28        0.5 2066  0.574653  0.577672      0.055127 0.712201            0.579640            0.581417                0.055161        0.732624


,epoch,loss,macro_f1,micro_f1,hamming_loss,lrap
0,1,0.260184,0.000000,0.000000,0.068421,0.420632
1,2,0.192669,0.127548,0.289279,0.063027,0.608429
2,3,0.156210,0.333436,0.452406,0.054695,0.681302
3,4,0.131014,0.475302,0.530221,0.052534,0.696628
4,5,0.112536,0.502938,0.546315,0.051393,0.706455
5,6,0.098798,0.524788,0.558728,0.051825,0.716550
6,7,0.085965,0.521662,0.544896,0.053796,0.698111
7,8,0.075689,0.571404,0.583042,0.053554,0.714310
8,9,0.066195,0.562262,0.574762,0.054816,0.707991
9,10,0.057894,0.554676,0.563578,0.056839,0.693955


## 9. Kết quả và so sánh công bằng
Bảng 6 dòng lấy điểm 27 nhãn thật từ reports; nếu thiếu sẽ ghi missing, không dựng số liệu.
Các cột metric chính của 28 nhãn chấm cả Neutral; **emotion27_*** chấm cùng 27 cảm xúc để đối chiếu.
Không diễn giải việc đổi tập nhãn là cải thiện. Điều kiện khác config/runtime/data và xác minh ID/target
được ghi trong `conditions`; nguồn 27 nhãn chỉ đọc, không ghi đè. LRAP của 27 nhãn cho neutral-only
(all-zero target) điểm 1 là một khác biệt cần nêu khi so với LRAP 28 nhãn.


In [14]:
run28('report')
for name in ('metrics_summary.csv', 'neutral_analysis.csv', 'comparison_27_vs_28.csv',
             'emotion27_per_class_comparison.csv', 'per_class_metrics.csv'):
    print(name)
    display(pd.read_csv(OUT/name))
from IPython.display import Markdown
display(Markdown((OUT/'experiment_report.md').read_text(encoding='utf-8')))


   model split  label_count  threshold    n  macro_f1  micro_f1  hamming_loss     lrap  emotion27_macro_f1  emotion27_micro_f1  emotion27_hamming_loss  emotion27_lrap
tfidf_lr   val           28        0.5 2066  0.231040  0.266160      0.060054 0.609645            0.238825            0.271238                0.060593        0.640745
     gru   val           28        0.5 2066  0.399227  0.459095      0.058861 0.605071            0.410764            0.465944                0.059177        0.636809
 phobert   val           28        0.5 2066  0.574653  0.577672      0.055127 0.712201            0.579640            0.581417                0.055161        0.732624
metrics_summary.csv


,model,split,label_count,threshold,n,macro_f1,micro_f1,hamming_loss,lrap,emotion27_macro_f1,emotion27_micro_f1,emotion27_hamming_loss,emotion27_lrap
0,tfidf_lr,val,28,0.5,2066,0.231040,0.266160,0.060054,0.609645,0.238825,0.271238,0.060593,0.640745
1,gru,val,28,0.5,2066,0.399227,0.459095,0.058861,0.605071,0.410764,0.465944,0.059177,0.636809
2,phobert,val,28,0.5,2066,0.574653,0.577672,0.055127,0.712201,0.579640,0.581417,0.055161,0.732624


neutral_analysis.csv


,model,split,label_id,label,precision,recall,f1,support,predicted_positive
0,tfidf_lr,val,27,neutral,0.500000,0.010638,0.020833,94,2
1,gru,val,27,neutral,0.250000,0.053191,0.087719,94,20
2,phobert,val,27,neutral,0.415094,0.468085,0.440000,94,106


comparison_27_vs_28.csv


,model,model_key,label_count,split,threshold,status,source,conditions,macro_f1,micro_f1,hamming_loss,lrap,emotion27_macro_f1,emotion27_micro_f1,emotion27_hamming_loss,emotion27_lrap,paired_validation_verified
0,TF-IDF + Logistic Regression,tfidf_lr,27,val,0.5,recorded,/kaggle/working/ViGoEmotions28/reports/tables/...,27-label metrics exclude Neutral; 28-label Mac...,0.238825,0.271238,0.060593,0.640745,0.238825,0.271238,0.060593,0.640745,True
1,TF-IDF + Logistic Regression,tfidf_lr,28,val,0.5,completed,/kaggle/working/ViGoEmotions28/reports/neutral...,Primary: preserve original annotations,0.231040,0.266160,0.060054,0.609645,0.238825,0.271238,0.060593,0.640745,NaN
2,BiGRU,gru,27,val,0.5,recorded,/kaggle/working/ViGoEmotions28/reports/tables/...,27-label metrics exclude Neutral; 28-label Mac...,0.406002,0.452609,0.060360,0.632236,0.406002,0.452609,0.060360,0.632236,True
3,BiGRU,gru,28,val,0.5,completed,/kaggle/working/ViGoEmotions28/reports/neutral...,Primary: preserve original annotations,0.399227,0.459095,0.058861,0.605071,0.410764,0.465944,0.059177,0.636809,NaN
4,PhoBERT,phobert,27,val,0.5,recorded,/kaggle/working/ViGoEmotions28/reports/tables/...,27-label metrics exclude Neutral; 28-label Mac...,0.439320,0.516045,0.053261,0.715767,0.439320,0.516045,0.053261,0.715767,True
5,PhoBERT,phobert,28,val,0.5,completed,/kaggle/working/ViGoEmotions28/reports/neutral...,Primary: preserve original annotations,0.574653,0.577672,0.055127,0.712201,0.579640,0.581417,0.055161,0.732624,NaN


emotion27_per_class_comparison.csv


,model,label_id,label,f1_27,f1_28,delta_f1
0,tfidf_lr,0,amusement,0.208877,0.208877,-5.551115e-17
1,tfidf_lr,1,excitement,0.042553,0.042553,0.000000e+00
2,tfidf_lr,2,joy,0.252525,0.252525,-5.551115e-17
3,tfidf_lr,3,love,0.298343,0.298343,0.000000e+00
4,tfidf_lr,4,desire,0.056075,0.056075,-8.326673e-17
...,...,...,...,...,...,...
76,phobert,22,grief,0.582278,0.640884,5.860550e-02
77,phobert,23,disgust,0.345133,0.471591,1.264582e-01
78,phobert,24,anger,0.549575,0.533708,-1.586721e-02
79,phobert,25,annoyance,0.555195,0.557632,2.437594e-03


per_class_metrics.csv


,model,split,label_id,label,precision,recall,f1,support,predicted_positive
0,tfidf_lr,val,0,amusement,0.714286,0.122324,0.208877,327,56
1,tfidf_lr,val,1,excitement,0.666667,0.021978,0.042553,91,3
2,tfidf_lr,val,2,joy,0.833333,0.148810,0.252525,168,30
3,tfidf_lr,val,3,love,0.818182,0.182432,0.298343,148,33
4,tfidf_lr,val,4,desire,0.600000,0.029412,0.056075,102,5
...,...,...,...,...,...,...,...,...,...
79,phobert,val,23,disgust,0.441489,0.506098,0.471591,164,188
80,phobert,val,24,anger,0.555556,0.513514,0.533708,185,171
81,phobert,val,25,annoyance,0.575563,0.540785,0.557632,331,311
82,phobert,val,26,disapproval,0.325153,0.430894,0.370629,123,163


# Thí nghiệm Neutral 28 nhãn

## 1. Mục đích và giao thức
Thêm Neutral (ID 27) vào 27 cảm xúc IDs 0–26. Không thay thế thí nghiệm 27 nhãn.
Seed 42; ngưỡng cố định 0.5. Giữ split và quy tắc loại train trùng heldout của project.
Văn bản NFC/chuẩn hóa khoảng trắng như `common.normalize`; TF-IDF và neural giữ hyperparameters từ config baseline.
BiGRU 10 epoch; PhoBERT 12 epoch.
Chọn epoch bằng validation Macro-F1 trên 28 nhãn; hòa chọn epoch sớm nhất.
Đổi tập nhãn chọn checkpoint là hệ quả của 28 đầu ra và được nêu rõ khi so sánh baseline 27 nhãn.

## 2. Mã hóa Neutral và bất nhất annotation
Vector multi-hot có đúng 28 phần tử. [27] → chỉ phần tử 27 bằng 1;
[2,27] → cả joy và neutral bằng 1. ID lặp được gộp, annotation gốc không bị viết lại.
Theo guideline do người dùng cung cấp, Neutral không đi cùng cảm xúc, nhưng dữ liệu được báo có
725 mẫu Neutral-only và 306 mẫu mixed. Thí nghiệm chính giữ mọi nhãn gốc, không giả định nhãn nào sai.
Số liệu thực đọc từ CSV (trước loại train overlap):

| split | raw_samples | neutral_only | mixed_neutral | neutral_positive |
| --- | --- | --- | --- | --- |
| train | 16531 | 570 | 251 | 821 |
| val | 2066 | 75 | 19 | 94 |
| test | 2067 | 80 | 36 | 116 |

## 3. Kết quả ba mô hình (số đo thật, ô trống nghĩa là chưa chạy)
| model | split | label_count | threshold | n | macro_f1 | micro_f1 | hamming_loss | lrap | emotion27_macro_f1 | emotion27_micro_f1 | emotion27_hamming_loss | emotion27_lrap |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| tfidf_lr | val | 28 | 0.500000 | 2066 | 0.231040 | 0.266160 | 0.060054 | 0.609645 | 0.238825 | 0.271238 | 0.060593 | 0.640745 |
| gru | val | 28 | 0.500000 | 2066 | 0.399227 | 0.459095 | 0.058861 | 0.605071 | 0.410764 | 0.465944 | 0.059177 | 0.636809 |
| phobert | val | 28 | 0.500000 | 2066 | 0.574653 | 0.577672 | 0.055127 | 0.712201 | 0.579640 | 0.581417 | 0.055161 | 0.732624 |

## 4. So sánh 27 và 28 nhãn
Macro/Micro-F1, Hamming và LRAP trên 27 nhãn và 28 nhãn có tập đầu ra khác nhau, không coi delta đó
là bằng chứng trực tiếp tốt hơn. Các cột emotion27_* chiếu dự đoán 28 nhãn về cùng 27 cảm xúc.
Baseline được đọc từ artifact thật; không dựng lại kết quả thiếu. Baseline chưa có test không được so với test 28 nhãn.

| model | model_key | label_count | split | threshold | status | source | conditions | macro_f1 | micro_f1 | hamming_loss | lrap | emotion27_macro_f1 | emotion27_micro_f1 | emotion27_hamming_loss | emotion27_lrap | paired_validation_verified |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| TF-IDF + Logistic Regression | tfidf_lr | 27 | val | 0.500000 | recorded | /kaggle/working/ViGoEmotions28/reports/tables/tfidf_lr_metrics.csv | 27-label metrics exclude Neutral; 28-label Macro-F1 has a different label set; Model settings match; Baseline checkpoint selected by 27-label Macro-F1; new checkpoint by 28-label Macro-F1; Different or unverified hardware/device; Runtime/source difference: python; Runtime/source difference: platform; Runtime/source difference: packages; IDs, emotion targets and recorded metrics verified | 0.238825 | 0.271238 | 0.060593 | 0.640745 | 0.238825 | 0.271238 | 0.060593 | 0.640745 | True |
| TF-IDF + Logistic Regression | tfidf_lr | 28 | val | 0.500000 | completed | /kaggle/working/ViGoEmotions28/reports/neutral_28_labels/models/tfidf_lr | Primary: preserve original annotations | 0.231040 | 0.266160 | 0.060054 | 0.609645 | 0.238825 | 0.271238 | 0.060593 | 0.640745 | nan |
| BiGRU | gru | 27 | val | 0.500000 | recorded | /kaggle/working/ViGoEmotions28/reports/tables/gru_metrics.csv | 27-label metrics exclude Neutral; 28-label Macro-F1 has a different label set; Model settings match; Baseline checkpoint selected by 27-label Macro-F1; new checkpoint by 28-label Macro-F1; Runtime/source difference: python; Runtime/source difference: platform; Runtime/source difference: packages; IDs, emotion targets and recorded metrics verified | 0.406002 | 0.452609 | 0.060360 | 0.632236 | 0.406002 | 0.452609 | 0.060360 | 0.632236 | True |
| BiGRU | gru | 28 | val | 0.500000 | completed | /kaggle/working/ViGoEmotions28/reports/neutral_28_labels/models/gru | Primary: preserve original annotations | 0.399227 | 0.459095 | 0.058861 | 0.605071 | 0.410764 | 0.465944 | 0.059177 | 0.636809 | nan |
| PhoBERT | phobert | 27 | val | 0.500000 | recorded | /kaggle/working/ViGoEmotions28/reports/tables/phobert_metrics.csv | 27-label metrics exclude Neutral; 28-label Macro-F1 has a different label set; DIFFERENT settings: phobert; Baseline checkpoint selected by 27-label Macro-F1; new checkpoint by 28-label Macro-F1; Runtime/source difference: python; Runtime/source difference: platform; Runtime/source difference: packages; IDs, emotion targets and recorded metrics verified | 0.439320 | 0.516045 | 0.053261 | 0.715767 | 0.439320 | 0.516045 | 0.053261 | 0.715767 | True |
| PhoBERT | phobert | 28 | val | 0.500000 | completed | /kaggle/working/ViGoEmotions28/reports/neutral_28_labels/models/phobert | Primary: preserve original annotations | 0.574653 | 0.577672 | 0.055127 | 0.712201 | 0.579640 | 0.581417 | 0.055161 | 0.732624 | nan |

## 5. Neutral precision, recall, F1 và số lượng
Support là số mẫu có Neutral thật, bao gồm mixed; predicted_positive là số mẫu được dự đoán Neutral.
Pipeline 27 nhãn không có đầu ra Neutral, nên Neutral metric baseline là không xác định; không tự gán bằng 0.

| model | split | label_id | label | precision | recall | f1 | support | predicted_positive |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| tfidf_lr | val | 27 | neutral | 0.500000 | 0.010638 | 0.020833 | 94 | 2 |
| gru | val | 27 | neutral | 0.250000 | 0.053191 | 0.087719 | 94 | 20 |
| phobert | val | 27 | neutral | 0.415094 | 0.468085 | 0.440000 | 94 | 106 |

## 6. Ảnh hưởng đến 27 cảm xúc
- TF-IDF + Logistic Regression: delta Macro-F1 trên cùng 27 cảm xúc = +0.000000; đọc các khác biệt điều kiện trong bảng so sánh trước khi quy nguyên nhân cho Neutral.
- BiGRU: delta Macro-F1 trên cùng 27 cảm xúc = +0.004762; đọc các khác biệt điều kiện trong bảng so sánh trước khi quy nguyên nhân cho Neutral.
- PhoBERT: delta Macro-F1 trên cùng 27 cảm xúc = +0.140321; đọc các khác biệt điều kiện trong bảng so sánh trước khi quy nguyên nhân cho Neutral.

Xem emotion27_per_class_comparison.csv (chỉ tạo delta khi xác minh ID/target/metric baseline)
và per_class_metrics.csv cho F1 đủ 28 nhãn. Không chỉ dựa việc có thể dự đoán Neutral để kết luận tốt hơn.

## 7. Inference và các loại dự đoán
Inference nạp model đã lưu, không hardcode kết quả. prediction_examples.csv chỉ chứa comment ví dụ do người dùng
nhập; validation_prediction_cases.csv chỉ chứa ID và số ca tìm được. Nếu model không dự đoán loại Neutral-only,
single-emotion, multi-emotion hoặc mixed nào thì báo không tìm thấy, không dựng ví dụ.

## 8. Giới hạn
Một seed/split; annotation mixed có thể nhiễu hoặc phản ánh mơ hồ; LRAP 27 nhãn của project cho all-zero target
điểm 1, còn Neutral-only 28 nhãn có target dương nên LRAP toàn bộ không trực tiếp tương đương.
TF-IDF OvR fit độc lập nên thêm nhãn Neutral dự kiến không đổi 27 classifier cảm xúc khi cùng môi trường/config;
neural chia sẻ backbone nên có thể cải thiện hoặc xung đột gradient, cần số đo để đánh giá.
Không có validation loss trong pipeline gốc nên không kết luận overfitting chỉ từ train loss.
Khác GPU/package/dữ liệu/epoch được ghi trong conditions; checksum CSV khác có thể chỉ do định dạng,
vì vậy kiểm tra thêm ID/target trước khi so sánh. Không bảo đảm bitwise-identical giữa runtime.
Test chỉ đánh giá qua bước khóa riêng, không dùng fit/chọn epoch/ngưỡng. Sau test không retrain trong study này.
Checkpoint lưu trọng số để inference, không có optimizer state để resume giữa epoch.
Toàn bộ output/checkpoint/vocabulary cần giữ private. Không xuất comment nguồn từ dataset.


## 10. Inference trên comment mới, từ checkpoint đã lưu
Bạn có thể thay danh sách COMMENTS. Đây là câu ví dụ tự viết, không khẳng định trước model sẽ dự đoán loại nào.
Mỗi model hiển thị văn bản gốc, cảm xúc dự đoán, Neutral, xác suất cho mọi nhãn vượt 0.5 và top 5 kể cả
không nhãn nào vượt ngưỡng. Không ép Neutral loại trừ cảm xúc: primary study cho phép đồng dự đoán.
Checkpoint pickle/joblib chỉ nạp từ nguồn của chính bạn. File ví dụ chứa text nhập vào nên cũng giữ private.


In [15]:
COMMENTS = [
    'Cuộc họp bắt đầu lúc tám giờ sáng thứ hai.',
    'Mình vui quá, cuối cùng cũng làm được rồi!',
    'Vừa vui vì gặp lại bạn, vừa buồn vì sắp phải chia tay.',
    'Mình hiểu rồi, nhưng vẫn hơi thất vọng về kết quả.',
]
summary, details = n28.infer_examples(OUT, COMMENTS)
display(summary)
for (model, index), table in details.groupby(['model', 'input_index'], sort=False):
    print(model, '| Input:', COMMENTS[index])
    display(table[['label', 'probability', 'predicted', 'top5']])


I0000 00:00:1793804285.264446      50 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


2026-11-04 14:58:06 INFO  WordSegmenter:24 - Loading Word Segmentation model


,input_index,text,predicted_emotions,neutral_predicted,predicted_probabilities,top5,model
0,0,Cuộc họp bắt đầu lúc tám giờ sáng thứ hai.,,False,{},"{""sadness"": 0.2516432644002463, ""amusement"": 0...",tfidf_lr
1,1,"Mình vui quá, cuối cùng cũng làm được rồi!",joy,False,"{""joy"": 0.5166882866964184}","{""joy"": 0.5166882866964184, ""amusement"": 0.262...",tfidf_lr
2,2,"Vừa vui vì gặp lại bạn, vừa buồn vì sắp phải c...",sadness,False,"{""sadness"": 0.7211367494953641}","{""sadness"": 0.7211367494953641, ""joy"": 0.21083...",tfidf_lr
3,3,"Mình hiểu rồi, nhưng vẫn hơi thất vọng về kết ...",,False,{},"{""disappointment"": 0.4422015476026205, ""sadnes...",tfidf_lr
4,0,Cuộc họp bắt đầu lúc tám giờ sáng thứ hai.,disappointment;sadness,False,"{""disappointment"": 0.8010769486427307, ""sadnes...","{""sadness"": 0.9804045557975769, ""disappointmen...",gru
5,1,"Mình vui quá, cuối cùng cũng làm được rồi!",amusement;joy,False,"{""amusement"": 0.6931966543197632, ""joy"": 0.956...","{""joy"": 0.9563148021697998, ""amusement"": 0.693...",gru
6,2,"Vừa vui vì gặp lại bạn, vừa buồn vì sắp phải c...",joy,False,"{""joy"": 0.7936869859695435}","{""joy"": 0.7936869859695435, ""amusement"": 0.296...",gru
7,3,"Mình hiểu rồi, nhưng vẫn hơi thất vọng về kết ...",sadness,False,"{""sadness"": 0.8087087273597717}","{""sadness"": 0.8087087273597717, ""remorse"": 0.2...",gru
8,0,Cuộc họp bắt đầu lúc tám giờ sáng thứ hai.,,True,"{""neutral"": 0.9819470643997192}","{""neutral"": 0.9819470643997192, ""annoyance"": 0...",phobert
9,1,"Mình vui quá, cuối cùng cũng làm được rồi!",excitement;joy;relief,False,"{""excitement"": 0.7662295699119568, ""joy"": 0.99...","{""joy"": 0.9946887493133545, ""excitement"": 0.76...",phobert


tfidf_lr | Input: Cuộc họp bắt đầu lúc tám giờ sáng thứ hai.


,label,probability,predicted,top5
0,sadness,0.251643,False,True
1,amusement,0.153468,False,True
2,joy,0.107730,False,True
3,disappointment,0.094347,False,True
4,annoyance,0.088256,False,True


tfidf_lr | Input: Mình vui quá, cuối cùng cũng làm được rồi!


... (17 outputs left out)

phobert | Input: Vừa vui vì gặp lại bạn, vừa buồn vì sắp phải chia tay.


,label,probability,predicted,top5
50,sadness,0.820377,True,True
51,joy,0.633174,True,True
52,amusement,0.348522,False,True
53,love,0.031928,False,True
54,disappointment,0.030534,False,True


phobert | Input: Mình hiểu rồi, nhưng vẫn hơi thất vọng về kết quả.


,label,probability,predicted,top5
55,disappointment,0.839789,True,True
56,remorse,0.821171,True,True
57,sadness,0.804740,True,True
58,realization,0.358997,False,True
59,approval,0.012880,False,True


### Tìm ví dụ thật cho từng kiểu dự đoán
Tìm trên xác suất validation đã lưu: Neutral-only, một cảm xúc, nhiều cảm xúc, Neutral + cảm xúc.
Nếu một kiểu không xuất hiện, báo `available_count=0`; không hardcode hay điều chỉnh threshold để tạo ca mong muốn.
DISPLAY_PRIVATE_TEXT mặc định bật để xem input và xác suất của ca tìm được trong runtime riêng tư.
CSV chỉ xuất ID/số lượng, không xuất comment gốc. Xóa output chứa text trước khi chia sẻ notebook.


In [16]:
DISPLAY_PRIVATE_TEXT = True  # Chỉ hiển thị trong runtime riêng tư; không chia sẻ output có comment nguồn.
cases = n28.find_validation_cases(OUT, DATA, display_text=DISPLAY_PRIVATE_TEXT)
display(cases)


tfidf_lr neutral_only : {'model': 'tfidf_lr', 'category': 'neutral_only', 'available_count': 2, 'sample_id': 'you001201'}
Private input: Cám ơn thầy, thở đi thôi- là có cả kho tàng :)))<br>3 cách vượt qua bất ổn tâm lý
<br>Cách 1: Phương pháp 5 nhìn 4 chạm 3 nghe 2 ngửi 1 nếm
<br>Cách 2: Tư duy 3%
<br>Cách 3: Phương pháp thở 4 thì
 input_index                                                                                                                                                                                                   text  label_id     label  probability  predicted  top5
           0 Cám ơn thầy, thở đi thôi- là có cả kho tàng :)))<br>3 cách vượt qua bất ổn tâm lý\n<br>Cách 1: Phương pháp 5 nhìn 4 chạm 3 nghe 2 ngửi 1 nếm\n<br>Cách 2: Tư duy 3%\n<br>Cách 3: Phương pháp thở 4 thì        27   neutral     0.602705       True  True
           0 Cám ơn thầy, thở đi thôi- là có cả kho tàng :)))<br>3 cách vượt qua bất ổn tâm lý\n<br>Cách 1: Phương pháp 5 nhìn 4 chạm 3 nghe 2 

ValueError: VM is already running, can't set classpath/options; VM started at  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.13/dist-packages/traitlets/config/application.py", line 1080, in launch_instance
    app.start()
  File "/usr/local/lib/python3.13/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.13/dist-packages/tornado/platform/asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "/usr/lib/python3.13/asyncio/base_events.py", line 684, in run_forever
    self._run_once()
  File "/usr/lib/python3.13/asyncio/base_events.py", line 2061, in _run_once
    handle._run()
  File "/usr/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._arg

## 11. Test cuối cùng — tùy chọn, sau khi khóa mọi lựa chọn
Mặc định **không chạy test**, tương thích protocol midterm 27 nhãn giữ test cho Final.
Nếu đã chốt model/hyperparameters/threshold, đặt RUN_FINAL_TEST=True. Cả ba model phải hoàn tất.
Bước này khóa checksum checkpoint và cấm train tiếp trong study, chỉ chấm test tại 0.5.
Không chọn mô hình/ngưỡng/epoch theo test. Nếu baseline 27 nhãn chỉ có val thì không tạo so sánh test giả.


In [17]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    run28('test')
    display(pd.read_csv(OUT/'metrics_summary.csv'))
else:
    print('Test chưa đánh giá. Chỉ dùng validation cho so sánh primary hiện tại.')


Test chưa đánh giá. Chỉ dùng validation cho so sánh primary hiện tại.


## 12. Lưu kết quả và checkpoint riêng tư

Chạy sau inference/report. ZIP toàn bộ study gồm CSV, NPZ, config, checkpoint,
tokenizer, vocabulary và segmenter; không chứa CSV/comment nguồn từ data/.
Tải ZIP từ file browser/Output trong /kaggle/working trước khi kết thúc session.
Giữ ZIP và notebook Private. Save & Run All sẽ chạy lại toàn bộ trong session mới.


In [19]:
import zipfile
archive_path = Path('/kaggle/working/vigoemotions_neutral28_private_results.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in OUT.rglob('*'):
        if path.is_file():
            archive.write(path, str(Path('neutral_28_labels')/path.relative_to(OUT)))
print('Private results:', archive_path, archive_path.stat().st_size/1024**2, 'MiB')
print('Tải ZIP từ file browser/Output:', archive_path)


Private results: /kaggle/working/vigoemotions_neutral28_private_results.zip 632.3364820480347 MiB
Tải ZIP từ file browser/Output: /kaggle/working/vigoemotions_neutral28_private_results.zip
